In [1]:
import numpy as np
from ase.io import read

DATA = (
    "/Users/mariospoulis/Documents/Materials Science and Engineering 2026-2027/Internship/umlff-for-molten-salts/data"
)
train_frames = read(DATA + "/Training.extxyz", index=":: 10")
print(f"training frames sampled: {len(train_frames)}")

elements = sorted({s for a in train_frames for s in a.get_chemical_symbols()})
print(f"elements present ({len(elements)}): {elements}")

epa = [a.get_potential_energy() / len(a) for a in train_frames]
print(f"energy/atom range (eV): {min(epa):.3f} to {max(epa):.3f}")

training frames sampled: 6840
elements present (12): ['Ba', 'Ca', 'Cl', 'Cs', 'K', 'Li', 'Mg', 'Na', 'Rb', 'Sr', 'Zn', 'Zr']
energy/atom range (eV): -5.117 to -1.838


In [ ]:
from uf3.data.composition import ChemicalSystem
from uf3.representation.bspline import BSplineBasis

cs = ChemicalSystem(element_list=elements, degree=2)
pairs = cs.interactions_map[2]
print(f"number of 2-body pairs: {len(pairs)}")

r_min_map = dict.fromkeys(pairs, 1.0)
r_max_map = dict.fromkeys(pairs, 6.0)
resolution_map = dict.fromkeys(pairs, 15)

bspline_config = BSplineBasis(
    cs,
    r_min_map=r_min_map,
    r_max_map=r_max_map,
    resolution_map=resolution_map,
)
print(f"total features (coefficients to fit): {bspline_config.n_feats}")

/Users/mariospoulis/Documents/Materials Science and Engineering 2026-2027/Internship/umlff-for-molten-salts/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


number of 2-body pairs: 78
total features (coefficients to fit): 1416


: 

In [ ]:
from uf3.data.io import DataCoordinator
from uf3.representation.process import BasisFeaturizer

for a in train_frames:
    a.info["energy"] = a.get_potential_energy()

data = DataCoordinator(energy_key="energy", force_key="force")
data.dataframe_from_lists(train_frames, prefix="t")
df_data = data.consolidate()
print(f"structures loaded: {len(df_data)}")

featurizer = BasisFeaturizer(bspline_config)
df_feats = featurizer.evaluate(df_data, energy_key="energy", progress="none")
print(f"feature table shape: {df_feats.shape}")

structures loaded: 6840


In [ ]:
from uf3.regression.least_squares import WeightedLinearModel, dataframe_to_tuples

regularizer = bspline_config.get_regularization_matrix(
    ridge_map={1: 1e-6, 2: 1e-4},
    curvature_map={2: 1e-6},
)

model = WeightedLinearModel(bspline_config, regularizer=regularizer)

x_e, y_e, x_f, y_f = dataframe_to_tuples(df_feats)

model.fit(x_e, y_e, x_f, y_f, weight=0.3)

print("fit complete")
print("coefficients finite:", np.isfinite(model.coefficients).all())

In [ ]:
from uf3.forcefield.calculator import UFCalculator

calc_uf3 = UFCalculator(model)


def evaluate(calc, path, stride=20):
    frames = read(path, index=f"::{stride}")
    de, fp, fr = [], [], []
    for a in frames:
        e_ref, f_ref = a.get_potential_energy(), a.arrays["force"]
        m = a.copy()
        m.calc = calc
        de.append((m.get_potential_energy() - e_ref) / len(m))
        fp.append(m.get_forces().ravel())
        fr.append(f_ref.ravel())
    emae = np.mean(np.abs(de)) * 1000
    frmse = np.sqrt(np.mean((np.concatenate(fp) - np.concatenate(fr)) ** 2)) * 1000
    return len(frames), emae, frmse


tests = [
    ("Test 1 (ternary)", DATA + "/Testing_2.extxyz"),
    ("Test 2 (multicomp)", DATA + "/Tessting_1.extxyz"),
]
for name, path in tests:
    n, emae, frmse = evaluate(calc_uf3, path)
    print(f"{name:20s} n={n:4d}   energy MAE {emae:7.2f} meV/atom   force RMSE {frmse:7.2f} meV/Å")

In [ ]:
for name, path in tests:
    n, emae, frmse = evaluate(calc_uf3, path, stride=5)
    print(f"{name:20s} n={n:4d}   energy MAE {emae:7.2f} meV/atom   force RMSE {frmse:7.2f} meV/Å")